In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-user-graduate" style="color:#18ab4b"></i> BYOC training — CatBoost credit model

Build a <i class="fas fa-file-code" style="color:#6c63ff"></i> **bring-your-own-container** (CatBoost), train it with SageMaker local mode (`instance_type='local'` — a real Docker training job), and store the <i class="fas fa-brain" style="color:#18ab4b"></i> model in oblako's <i class="fas fa-database" style="color:#18ab4b"></i> **S3Proxy**. Mirrors `sagemaker_endpoints/ml_models`.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## <i class="fas fa-file-code" style="color:#6c63ff"></i> Build the container + <i class="fas fa-user-graduate" style="color:#18ab4b"></i> train

In [2]:
import os
import tempfile

import creditdata
from sagemaker.estimator import Estimator

from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

IMAGE = "oblako-credit-risk:latest"
HP = {
    "iterations": "150",
    "depth": "6",
    "learning-rate": "0.1",
    "factor": "20",
    "offset": "600",
}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # CatBoost BYOC (./container)

data_dir = tempfile.mkdtemp(prefix="credit-train-")
creditdata.write_training_csv(os.path.join(data_dir, "train.csv"), n=600)
out_dir = tempfile.mkdtemp(prefix="credit-out-")

estimator = Estimator(
    image_uri=IMAGE,
    role="arn:aws:iam::000000000000:role/dummy",
    instance_count=1,
    instance_type="local",
    sagemaker_session=sm.get_session(),
    output_path=f"file://{out_dir}",
    hyperparameters=HP,
)
estimator.fit({"train": f"file://{data_dir}"})

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/deburky/Library/Application Support/sagemaker/config.yaml


INFO:sagemaker:Creating training-job with name: oblako-credit-risk-2026-05-30-14-35-29-024


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.


...

INFO:sagemaker.local.image:docker command: docker-compose -f /private/tmp/claude-501/tmppu15p47_/docker-compose.yaml up --build --abort-on-container-exit


Step 1/7 : FROM python:3.11-slim
 ---> a3ab0b966bc4
Step 2/7 : RUN apt-get update && apt-get install -y --no-install-recommends libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> dac6d10a3439
Step 3/7 : RUN pip install --no-cache-dir catboost numpy pandas joblib
 ---> Using cache
 ---> 8432245a91ff
Step 4/7 : COPY src/ /opt/program/
 ---> Using cache
 ---> 66f061362b95
Step 5/7 : WORKDIR /opt/program
 ---> Using cache
 ---> b6c9560c6fd2
Step 6/7 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> e724b186fc1d
Step 7/7 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> 2f70f8506e14
Successfully built 2f70f8506e14
Successfully tagged oblako-credit-risk:latest
time="2026-05-30T16:35:29+02:00" level=warning msg="/private/tmp/claude-501/tmppu15p47_/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"


time="2026-05-30T16:35:29+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmppu15p47_\".\nSet `external: true` to use an existing network"
 Container seodxgvk20-algo-1-8fiy9 Creating 
 Container seodxgvk20-algo-1-8fiy9 Created 
Attaching to seodxgvk20-algo-1-8fiy9
 Container seodxgvk20-algo-1-8fiy9 Starting 
 Container seodxgvk20-algo-1-8fiy9 Started 


INFO:sagemaker.local.image:===== Job Complete =====


seodxgvk20-algo-1-8fiy9  | trained CatBoost on 600 rows, 12 features (default rate 0.39)
seodxgvk20-algo-1-8fiy9 exited with code 0
 Compose Stopping Aborting on container exit...
 Container seodxgvk20-algo-1-8fiy9 Stopping 
 Container seodxgvk20-algo-1-8fiy9 Stopped 


## <i class="fas fa-database" style="color:#18ab4b"></i> Store the trained model in oblako's S3

In [3]:
import glob

from oblako.services import S3ProxyService

model_tar = glob.glob(os.path.join(out_dir, "**", "model.tar.gz"), recursive=True)[0]
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="credit-risk-models")
except Exception:
    pass
s3.upload_file(model_tar, "credit-risk-models", "credit-risk/model.tar.gz")
print("stored: s3://credit-risk-models/credit-risk/model.tar.gz")
print(
    "bucket:",
    [
        o["Key"]
        for o in s3.list_objects_v2(Bucket="credit-risk-models").get("Contents", [])
    ],
)

stored: s3://credit-risk-models/credit-risk/model.tar.gz
bucket: ['credit-risk/model.tar.gz']
